# Auditoria — BRACOFER resumo_fatura_fechamento_padrao.xlsx

Verifica cruzando o arquivo de fechamento gerado pelo notebook de normalização
contra o arquivo normalizado intermediário (fonte). Cobre 7 grupos de verificação:

- **G1** Contagem e completude
- **G2** Integridade dos valores financeiros
- **G3** Datas (emissao, vencimento, pagamento)
- **G4** Centro de custo — fallback silencioso
- **G5** Código da conta
- **G6** Fornecedor / credor
- **G7** Filial

In [1]:
from pathlib import Path
import pandas as pd
import json

REFS_DIR = Path("../../02-Referencias")
BRACOFER_DIR = REFS_DIR / "Bracofer"

ARQUIVO_NORMALIZADO = BRACOFER_DIR / "Relatorio Contas a Pagar 03-2026 - Normalizado.xlsx"
ARQUIVO_FECHAMENTO  = BRACOFER_DIR / "BRACOFER - resumo_fatura_fechamento_padrao.xlsx"

for f in [ARQUIVO_NORMALIZADO, ARQUIVO_FECHAMENTO]:
    status = "OK" if f.exists() else "NAO ENCONTRADO"
    print(f"{status}: {f.resolve()}")

OK: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Relatorio Contas a Pagar 03-2026 - Normalizado.xlsx
OK: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\BRACOFER - resumo_fatura_fechamento_padrao.xlsx


In [2]:
# --- Carrega os dois arquivos ---
df_norm = pd.read_excel(ARQUIVO_NORMALIZADO, dtype=str)
df_fech = pd.read_excel(ARQUIVO_FECHAMENTO,  dtype=str)

# Normaliza nomes de colunas
df_norm.columns = [c.strip().lower().replace(' ', '_') for c in df_norm.columns]
df_fech.columns = [c.strip().lower().replace(' ', '_') for c in df_fech.columns]

print("=== Normalizado ===")
print(f"Linhas: {len(df_norm)}")
print("Colunas:", df_norm.columns.tolist())
print()
print("=== Fechamento ===")
print(f"Linhas: {len(df_fech)}")
print("Colunas:", df_fech.columns.tolist())

=== Normalizado ===
Linhas: 265
Colunas: ['numero_documento', 'descricao_documento', 'observacao', 'fornecedor', 'emissao', 'vencimento', 'pagamento', 'valor_documento', 'desconto', 'juros', 'multa', 'devolucao', 'liquido', 'empresa', 'classificacao_financeira_codigo', 'classificacao_financeira_descricao', 'classificacao_financeira_valor', 'centro_custos_descricao', 'centro_custos_valor']

=== Fechamento ===
Linhas: 265
Colunas: ['id', 'segmento', 'n1_cod_centro_custo', 'n1_centro_custo', 'n1_cc', 'n2_cod_centro_custo', 'n2_centro_custo', 'n2_cc', 'n3_cod_centro_custo', 'n3_centro_custo', 'n3_cc', 'n4_cod_centro_custo', 'n4_centro_custo', 'n4_cc', 'cod_conta', 'conta', 'cod_conta-descr', 'filial', 'titulo', 'valor_nf', 'valor_pago', 'valor_conta', 'observacao', 'data_nf', 'data_vecto', 'data_pagamento', 'cod_credor_forn_cli_func', 'credor_forn_cli_func', 'origem', 'sistema', 'dados_auxiliares', 'valor_oficial', 'de-para1', 'de-para2', 'custeio_variável']


In [3]:
# Utilitario de parse de valor BRL
def brl_to_float(v):
    if pd.isna(v):
        return float('nan')
    s = str(v).strip().replace('R$', '').replace(' ', '')
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return float('nan')

# Contador de alertas por grupo
alertas = {}

print("Utilitarios carregados.")

Utilitarios carregados.


## G1 — Contagem e Completude

In [4]:
n_norm = len(df_norm)
n_fech = len(df_fech)
dif_linhas = n_fech - n_norm

print(f"Linhas no normalizado: {n_norm}")
print(f"Linhas no fechamento:  {n_fech}")
print(f"Diferenca:             {dif_linhas:+d}")

# Titulos presentes no normalizado mas ausentes no fechamento
titulos_norm = set(df_norm['numero_documento'].dropna().str.strip())
titulos_fech = set(df_fech['titulo'].dropna().str.strip()) if 'titulo' in df_fech.columns else set()

ausentes_no_fech = titulos_norm - titulos_fech
extras_no_fech   = titulos_fech - titulos_norm

print(f"\nTitulos no normalizado ausentes no fechamento: {len(ausentes_no_fech)}")
if ausentes_no_fech:
    for t in sorted(ausentes_no_fech):
        print(f"  {t}")

print(f"\nTitulos extras no fechamento (nao existem no normalizado): {len(extras_no_fech)}")
if extras_no_fech:
    for t in sorted(extras_no_fech):
        print(f"  {t}")

alertas['G1'] = {
    'linhas_normalizado': n_norm,
    'linhas_fechamento': n_fech,
    'diferenca_linhas': dif_linhas,
    'titulos_ausentes_no_fechamento': sorted(ausentes_no_fech),
    'titulos_extras_no_fechamento': sorted(extras_no_fech),
}

Linhas no normalizado: 265
Linhas no fechamento:  265
Diferenca:             +0

Titulos no normalizado ausentes no fechamento: 0

Titulos extras no fechamento (nao existem no normalizado): 0


## G2 — Integridade dos Valores Financeiros

In [5]:
# Parse valores
df_norm['vd_num'] = df_norm['valor_documento'].apply(brl_to_float)
df_norm['liq_num'] = df_norm['liquido'].apply(brl_to_float)

df_fech['vnf_num']    = df_fech['valor_nf'].apply(brl_to_float)
df_fech['vpago_num']  = df_fech['valor_pago'].apply(brl_to_float)
df_fech['vconta_num'] = df_fech['valor_conta'].apply(brl_to_float)

soma_vdoc_norm  = df_norm['vd_num'].sum()
soma_liq_norm   = df_norm['liq_num'].sum()
soma_vnf_fech   = df_fech['vnf_num'].sum()
soma_vpago_fech = df_fech['vpago_num'].sum()

print(f"Soma valor_documento  (normalizado): R$ {soma_vdoc_norm:>12,.2f}")
print(f"Soma valor_nf         (fechamento):  R$ {soma_vnf_fech:>12,.2f}")
print(f"Diferenca valor_nf vs valor_documento: R$ {soma_vnf_fech - soma_vdoc_norm:+,.2f}")
print()
print(f"Soma liquido  (normalizado): R$ {soma_liq_norm:>12,.2f}")
print(f"Soma valor_pago (fechamento): R$ {soma_vpago_fech:>12,.2f}")
print(f"Diferenca valor_pago vs liquido: R$ {soma_vpago_fech - soma_liq_norm:+,.2f}")

# Verificar valor_conta == valor_pago (linha a linha)
dif_conta_pago = df_fech[abs(df_fech['vconta_num'] - df_fech['vpago_num']) > 0.01]
print(f"\nLinhas onde valor_conta != valor_pago (tolerancia 0.01): {len(dif_conta_pago)}")
if not dif_conta_pago.empty:
    print(dif_conta_pago[['titulo', 'valor_nf', 'valor_pago', 'valor_conta']].to_string())

# Verificar valor_nf != valor_pago (titulos com desconto/diferenca)
dif_nf_pago = df_fech[abs(df_fech['vnf_num'] - df_fech['vpago_num']) > 0.01]
print(f"\nLinhas onde valor_nf != valor_pago (desconto ou diferenca real): {len(dif_nf_pago)}")
if not dif_nf_pago.empty:
    print(dif_nf_pago[['titulo', 'valor_nf', 'valor_pago', 'credor_forn_cli_func']].head(10).to_string())

alertas['G2'] = {
    'soma_valor_documento_normalizado': round(soma_vdoc_norm, 2),
    'soma_valor_nf_fechamento': round(soma_vnf_fech, 2),
    'diferenca_valor_nf': round(soma_vnf_fech - soma_vdoc_norm, 2),
    'soma_liquido_normalizado': round(soma_liq_norm, 2),
    'soma_valor_pago_fechamento': round(soma_vpago_fech, 2),
    'diferenca_valor_pago': round(soma_vpago_fech - soma_liq_norm, 2),
    'linhas_valor_conta_diferente_valor_pago': len(dif_conta_pago),
    'linhas_valor_nf_diferente_valor_pago': len(dif_nf_pago),
}

Soma valor_documento  (normalizado): R$ 1,532,638.06
Soma valor_nf         (fechamento):  R$ 1,532,638.06
Diferenca valor_nf vs valor_documento: R$ +0.00

Soma liquido  (normalizado): R$ 1,532,503.86
Soma valor_pago (fechamento): R$ 1,532,503.86
Diferenca valor_pago vs liquido: R$ +0.00

Linhas onde valor_conta != valor_pago (tolerancia 0.01): 0

Linhas onde valor_nf != valor_pago (desconto ou diferenca real): 2
           titulo  valor_nf valor_pago                              credor_forn_cli_func
22  000027114-001    295,00     281,28  12622699000100 - POLIVIDA CLINICA MEDICA SS LTDA
31  000045363-001  1.959,00   1.838,52           00392500000116 - SOFTLAND SISTEMAS LTDA


## G3 — Datas

In [6]:
# Alinha as duas tabelas por posicao (mesma ordem de linhas esperada)
df_n = df_norm.reset_index(drop=True)
df_f = df_fech.reset_index(drop=True)

n_rows = min(len(df_n), len(df_f))

def _norm_date(v):
    s = str(v).strip() if pd.notna(v) else ''
    return s

erros_data_nf    = []
erros_data_vecto = []
erros_data_pago  = []
erros_pago_aberto = []  # pagamento="-" mas data_pagamento != "-"

for i in range(n_rows):
    titulo = _norm_date(df_f.loc[i, 'titulo']) if 'titulo' in df_f.columns else str(i)

    emissao = _norm_date(df_n.loc[i, 'emissao'])
    data_nf  = _norm_date(df_f.loc[i, 'data_nf']) if 'data_nf' in df_f.columns else ''
    if emissao and data_nf and emissao != data_nf:
        erros_data_nf.append({'linha': i+1, 'titulo': titulo, 'emissao_fonte': emissao, 'data_nf_fechamento': data_nf})

    vencimento = _norm_date(df_n.loc[i, 'vencimento'])
    data_vecto = _norm_date(df_f.loc[i, 'data_vecto']) if 'data_vecto' in df_f.columns else ''
    if vencimento and data_vecto and vencimento != data_vecto:
        erros_data_vecto.append({'linha': i+1, 'titulo': titulo, 'vencimento_fonte': vencimento, 'data_vecto_fechamento': data_vecto})

    pagamento   = _norm_date(df_n.loc[i, 'pagamento'])
    data_pago   = _norm_date(df_f.loc[i, 'data_pagamento']) if 'data_pagamento' in df_f.columns else ''
    if pagamento and data_pago and pagamento != data_pago:
        erros_data_pago.append({'linha': i+1, 'titulo': titulo, 'pagamento_fonte': pagamento, 'data_pagamento_fechamento': data_pago})
    
    # Titulos em aberto (pagamento="-") mas com data_pagamento preenchida
    if pagamento == '-' and data_pago not in ('', '-', 'nan', 'NaT', 'None'):
        erros_pago_aberto.append({'linha': i+1, 'titulo': titulo, 'pagamento_fonte': pagamento, 'data_pagamento_fechamento': data_pago})

print(f"Divergencias data_nf vs emissao:               {len(erros_data_nf)}")
print(f"Divergencias data_vecto vs vencimento:         {len(erros_data_vecto)}")
print(f"Divergencias data_pagamento vs pagamento:      {len(erros_data_pago)}")
print(f"Titulos em aberto com data_pagamento incorreta: {len(erros_pago_aberto)}")

if erros_data_nf:
    print("\n--- Erros data_nf ---")
    for e in erros_data_nf[:10]:
        print(e)

if erros_data_vecto:
    print("\n--- Erros data_vecto ---")
    for e in erros_data_vecto[:10]:
        print(e)

if erros_data_pago:
    print("\n--- Erros data_pagamento ---")
    for e in erros_data_pago[:10]:
        print(e)

if erros_pago_aberto:
    print("\n--- Titulos em aberto com data_pagamento incorreta ---")
    for e in erros_pago_aberto:
        print(e)

alertas['G3'] = {
    'erros_data_nf': len(erros_data_nf),
    'erros_data_vecto': len(erros_data_vecto),
    'erros_data_pagamento': len(erros_data_pago),
    'titulos_abertos_com_data_pago_incorreta': len(erros_pago_aberto),
    'detalhe_erros_data_pago': erros_data_pago[:20],
    'detalhe_titulos_abertos_incorretos': erros_pago_aberto,
}

Divergencias data_nf vs emissao:               0
Divergencias data_vecto vs vencimento:         0
Divergencias data_pagamento vs pagamento:      0
Titulos em aberto com data_pagamento incorreta: 0


## G4 — Centro de Custo (fallback silencioso)

In [7]:
cc_col_norm = 'centro_custos_descricao'
cc_col_fech = 'n4_cod_centro_custo' if 'n4_cod_centro_custo' in df_f.columns else 'n3_cod_centro_custo'

# Mapeamento: descricao do CC na fonte -> cod_cc no fechamento
mapa_cc = {}
fallbacks_incorretos = []   # descricao nao e 'Administrativo' mas foi para 1.3.1.1

for i in range(n_rows):
    desc = str(df_n.loc[i, cc_col_norm]).strip() if cc_col_norm in df_n.columns else ''
    cod  = str(df_f.loc[i, cc_col_fech]).strip() if cc_col_fech in df_f.columns else ''
    if desc not in mapa_cc:
        mapa_cc[desc] = {'cod_resultante': cod, 'ocorrencias': 0}
    mapa_cc[desc]['ocorrencias'] += 1

    desc_lower = desc.lower()
    if cod == '1.3.1.1' and 'administrativo' not in desc_lower:
        fallbacks_incorretos.append({
            'linha': i+1,
            'titulo': str(df_f.loc[i, 'titulo']) if 'titulo' in df_f.columns else '',
            'cc_descricao_fonte': desc,
            'cod_cc_no_fechamento': cod,
        })

print("=== Mapeamento de Centro de Custo (descricao fonte -> cod fechamento) ===")
for desc, info in sorted(mapa_cc.items()):
    print(f"  '{desc}' ({info['ocorrencias']}x) -> {info['cod_resultante']}")

print(f"\nFallbacks possivelmente incorretos (1.3.1.1 para descricao nao-Administrativo): {len(fallbacks_incorretos)}")
if fallbacks_incorretos:
    for e in fallbacks_incorretos[:20]:
        print(f"  Linha {e['linha']} | {e['titulo']} | CC fonte: '{e['cc_descricao_fonte']}' -> {e['cod_cc_no_fechamento']}")

alertas['G4'] = {
    'mapa_cc': [{**{'descricao': k}, **v} for k, v in mapa_cc.items()],
    'fallbacks_incorretos': len(fallbacks_incorretos),
    'detalhe_fallbacks': fallbacks_incorretos[:20],
}

=== Mapeamento de Centro de Custo (descricao fonte -> cod fechamento) ===
  'Administrativo' (154x) -> 1.3.1.1
  'Comercial' (104x) -> 1.3.1.2
  'Logistica' (3x) -> 1.3.1.4
  'Operacional' (4x) -> 1.3.1.3

Fallbacks possivelmente incorretos (1.3.1.1 para descricao nao-Administrativo): 0


## G5 — Código da Conta

In [8]:
erros_cod_conta = []
erros_cod_conta_descr = []

for i in range(n_rows):
    cod_fonte = str(df_n.loc[i, 'classificacao_financeira_codigo']).strip() if 'classificacao_financeira_codigo' in df_n.columns else ''
    cod_fech  = str(df_f.loc[i, 'cod_conta']).strip() if 'cod_conta' in df_f.columns else ''
    desc_fech = str(df_f.loc[i, 'conta']).strip() if 'conta' in df_f.columns else ''
    cod_descr_fech = str(df_f.loc[i, 'cod_conta-descr']).strip() if 'cod_conta-descr' in df_f.columns else ''
    titulo = str(df_f.loc[i, 'titulo']) if 'titulo' in df_f.columns else str(i)

    # Compara ignorando zeros a esquerda
    cod_fonte_norm = cod_fonte.lstrip('0') or '0'
    cod_fech_norm  = cod_fech.lstrip('0') or '0'
    if cod_fonte_norm != cod_fech_norm and cod_fonte and cod_fech and cod_fech not in ('nan', 'None', '<NA>', ''):
        erros_cod_conta.append({
            'linha': i+1, 'titulo': titulo,
            'cod_fonte': cod_fonte, 'cod_fechamento': cod_fech
        })

    # Verifica cod_conta-descr = cod_conta + ' ' + conta
    esperado = f"{cod_fech} {desc_fech}".strip()
    if cod_descr_fech and cod_descr_fech not in ('nan', 'None', '<NA>') and esperado and cod_descr_fech != esperado:
        erros_cod_conta_descr.append({
            'linha': i+1, 'titulo': titulo,
            'cod_conta-descr_real': cod_descr_fech,
            'esperado': esperado
        })

print(f"Divergencias cod_conta vs classificacao_financeira_codigo: {len(erros_cod_conta)}")
if erros_cod_conta:
    for e in erros_cod_conta[:10]:
        print(f"  Linha {e['linha']} | {e['titulo']} | fonte={e['cod_fonte']} | fechamento={e['cod_fechamento']}")

print(f"\nDivergencias cod_conta-descr vs cod+descricao: {len(erros_cod_conta_descr)}")
if erros_cod_conta_descr:
    for e in erros_cod_conta_descr[:10]:
        print(f"  Linha {e['linha']} | real='{e['cod_conta-descr_real']}' | esperado='{e['esperado']}'")

# Mapa de contas unicas
contas_unicas = df_f[['cod_conta', 'conta']].dropna().drop_duplicates()
print("\n=== Contas unicas no fechamento ===")
print(contas_unicas.to_string(index=False))

alertas['G5'] = {
    'erros_cod_conta': len(erros_cod_conta),
    'erros_cod_conta_descr': len(erros_cod_conta_descr),
    'contas_unicas': contas_unicas.to_dict('records'),
}

Divergencias cod_conta vs classificacao_financeira_codigo: 0

Divergencias cod_conta-descr vs cod+descricao: 0

=== Contas unicas no fechamento ===
cod_conta                      conta
  2010101                   Serviços
      301       Mercadoria p/Revenda
  2050201         Assistencia Medica
  2020103  Desp. e Materiais Diveros
  2040120                   Despesas
  2050206                Emprestimos
  2050109         Pensão Alimenticia
  2040101                    Aluguel
  2040208            Taxa Adm Cartão
    20305      Reembolso de Clientes
  2050101                    Salario
  2050203                  Refeições
  2020202 Impostos ,licenças e Taxas
  2060402                       FGTS
  2040201                Juros pagos
  2050108                  Rescisões
  2020205        Despesas de Viagens
  2060405            Seguro Trabalho
  2050204               Cesta Basica
  2060403                   Sindical
  2040202          Tarifas Bancarias


## G6 — Fornecedor / Credor

In [9]:
erros_fornecedor = []

for i in range(n_rows):
    forn_norm = str(df_n.loc[i, 'fornecedor']).strip() if 'fornecedor' in df_n.columns else ''
    cred_fech = str(df_f.loc[i, 'credor_forn_cli_func']).strip() if 'credor_forn_cli_func' in df_f.columns else ''
    titulo    = str(df_f.loc[i, 'titulo']) if 'titulo' in df_f.columns else str(i)

    if forn_norm and cred_fech and cred_fech not in ('nan', 'None', '<NA>') and forn_norm != cred_fech:
        erros_fornecedor.append({
            'linha': i+1, 'titulo': titulo,
            'fornecedor_fonte': forn_norm[:60],
            'credor_fechamento': cred_fech[:60]
        })

# Verificar cod_credor vazio
cod_cred_col = 'cod_credor_forn_cli_func'
if cod_cred_col in df_f.columns:
    vazios_cod_credor = df_f[df_f[cod_cred_col].isna() | df_f[cod_cred_col].isin(['', 'nan', 'None', '<NA>'])]
    print(f"Linhas com cod_credor_forn_cli_func vazio: {len(vazios_cod_credor)} de {len(df_f)}")
else:
    print("Coluna cod_credor_forn_cli_func nao encontrada no fechamento.")
    vazios_cod_credor = pd.DataFrame()

print(f"Divergencias fornecedor vs credor_forn_cli_func: {len(erros_fornecedor)}")
if erros_fornecedor:
    for e in erros_fornecedor[:10]:
        print(f"  Linha {e['linha']} | {e['titulo']}")
        print(f"    Fonte:     {e['fornecedor_fonte']}")
        print(f"    Fechamento: {e['credor_fechamento']}")

alertas['G6'] = {
    'vazios_cod_credor': len(vazios_cod_credor),
    'total_linhas': len(df_f),
    'erros_fornecedor': len(erros_fornecedor),
    'cod_credor_col_existe': cod_cred_col in df_f.columns,
}

Linhas com cod_credor_forn_cli_func vazio: 265 de 265
Divergencias fornecedor vs credor_forn_cli_func: 0


## G7 — Filial

In [10]:
empresa_unica = df_n['empresa'].unique().tolist() if 'empresa' in df_n.columns else []
filial_unica  = df_f['filial'].unique().tolist()  if 'filial'  in df_f.columns else []

print(f"Valores unicos de 'empresa' no normalizado: {empresa_unica}")
print(f"Valores unicos de 'filial' no fechamento:   {filial_unica}")

# Verifica coerencia: empresa '003' deve aparecer como '3' ou '003' no fechamento
empresas_sem_par = []
for emp in empresa_unica:
    emp_s = str(emp).strip().lstrip('0') or '0'
    match = any(str(f).strip().lstrip('0') == emp_s for f in filial_unica)
    if not match:
        empresas_sem_par.append(emp)

print(f"\nEmpresas sem correspondencia no fechamento (comparando sem zeros a esquerda): {empresas_sem_par}")

alertas['G7'] = {
    'empresa_normalizado': empresa_unica,
    'filial_fechamento': filial_unica,
    'empresas_sem_correspondencia': empresas_sem_par,
}

Valores unicos de 'empresa' no normalizado: ['003']
Valores unicos de 'filial' no fechamento:   ['3']

Empresas sem correspondencia no fechamento (comparando sem zeros a esquerda): []


## Resumo Final

In [11]:
def severidade(grupo, dados):
    """Classifica severidade com base no grupo e quantidade de problemas."""
    if grupo == 'G1':
        if dados['diferenca_linhas'] != 0 or dados['titulos_ausentes_no_fechamento']:
            return 'Alta'
        return 'OK'
    if grupo == 'G2':
        if abs(dados['diferenca_valor_nf']) > 0.02 or abs(dados['diferenca_valor_pago']) > 0.02:
            return 'Alta'
        if dados['linhas_valor_conta_diferente_valor_pago'] > 0:
            return 'Media'
        return 'OK'
    if grupo == 'G3':
        n = dados['erros_data_nf'] + dados['erros_data_vecto'] + dados['erros_data_pagamento']
        if dados['titulos_abertos_com_data_pago_incorreta'] > 0:
            return 'Alta'
        if n > 0:
            return 'Media'
        return 'OK'
    if grupo == 'G4':
        if dados['fallbacks_incorretos'] > 0:
            return 'Alta'
        return 'OK'
    if grupo == 'G5':
        if dados['erros_cod_conta'] > 0:
            return 'Alta'
        if dados['erros_cod_conta_descr'] > 0:
            return 'Baixa'
        return 'OK'
    if grupo == 'G6':
        if dados['erros_fornecedor'] > 0:
            return 'Alta'
        if dados['vazios_cod_credor'] > 0:
            return 'Baixa'
        return 'OK'
    if grupo == 'G7':
        if dados['empresas_sem_correspondencia']:
            return 'Media'
        return 'OK'
    return 'OK'

print("========================================")
print("  RESUMO DA AUDITORIA — BRACOFER 03/2026")
print("========================================")

descricoes = {
    'G1': 'Contagem e completude',
    'G2': 'Integridade financeira',
    'G3': 'Datas',
    'G4': 'Centro de custo',
    'G5': 'Codigo da conta',
    'G6': 'Fornecedor / Credor',
    'G7': 'Filial',
}

resumo_final = []
for grupo, dados in alertas.items():
    sev = severidade(grupo, dados)
    resumo_final.append({'grupo': grupo, 'descricao': descricoes[grupo], 'severidade': sev})
    print(f"  {grupo} | {descricoes[grupo]:<30} | {sev}")

alertas['_resumo'] = resumo_final
alertas['_totais'] = {
    'linhas_fechamento': n_fech,
    'linhas_normalizado': n_norm,
    'soma_total_valor_nf': round(soma_vnf_fech, 2),
}
print("\nJSON do sumario para canvas:")
print(json.dumps(alertas, indent=2, ensure_ascii=False, default=str))

  RESUMO DA AUDITORIA — BRACOFER 03/2026
  G1 | Contagem e completude          | OK
  G2 | Integridade financeira         | OK
  G3 | Datas                          | OK
  G4 | Centro de custo                | OK
  G5 | Codigo da conta                | OK
  G6 | Fornecedor / Credor            | Baixa
  G7 | Filial                         | OK

JSON do sumario para canvas:
{
  "G1": {
    "linhas_normalizado": 265,
    "linhas_fechamento": 265,
    "diferenca_linhas": 0,
    "titulos_ausentes_no_fechamento": [],
    "titulos_extras_no_fechamento": []
  },
  "G2": {
    "soma_valor_documento_normalizado": 1532638.06,
    "soma_valor_nf_fechamento": 1532638.06,
    "diferenca_valor_nf": 0.0,
    "soma_liquido_normalizado": 1532503.86,
    "soma_valor_pago_fechamento": 1532503.86,
    "diferenca_valor_pago": 0.0,
    "linhas_valor_conta_diferente_valor_pago": 0,
    "linhas_valor_nf_diferente_valor_pago": 2
  },
  "G3": {
    "erros_data_nf": 0,
    "erros_data_vecto": 0,
    "erros_data_p